# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nancy121205/flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will compare Logistic Regression and Random Forest. Logistic Regression provides a relatively simple baseline model, while Random Forest can learn nonlinear patterns and interactions among content and search-performance signals.

Because my task is ranking, I will sort pages by the predicted probability of the decline proxy. Precision@20 and Precision@50 will measure how many pages in the top of the queue match that proxy. The model will support human review, not automatically decide which pages to change.


In [1]:
RANDOM_STATE = 42
TEST_CLIENT_FRACTION = 0.20
K_VALUES = (20, 50)

print("Methods: Logistic Regression and Random Forest")
print("Ranking metrics: Precision@20 and Precision@50")
print("Random seed:", RANDOM_STATE)

Methods: Logistic Regression and Random Forest
Ranking metrics: Precision@20 and Precision@50
Random seed: 42


## 2. Split design

I will hold out approximately 20% of clients for testing. All pages belonging to a held-out client will remain outside the training set. This reduces the risk of the model appearing to perform well by learning patterns specific to clients it has already seen.

The split is grouped by client, not by individual row. Both the baseline and the learned models will be evaluated on exactly the same test rows. The random seed is fixed for reproducibility.

This is a client-generalization test, not a future-time test. Because the starter label describes the current trend rather than a future outcome, the results must not be interpreted as forecasting future declines.


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit

# Load the anonymized starter data.
data_url = (
    "https://raw.githubusercontent.com/"
    "flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)
raw = pd.read_csv(data_url)

required = [
    "content_id", "client_id", "impressions_90d",
    "sessions_90d", "content_age_days", "trend_direction",
]
missing = [c for c in required if c not in raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Match the starter pipeline's inclusion rules.
df = raw.copy()
df = df[
    (pd.to_numeric(df["impressions_90d"], errors="coerce") > 0)
    & (pd.to_numeric(df["content_age_days"], errors="coerce") >= 90)
].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Target is a proxy, not a future observed outcome.
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

if df["client_id"].nunique() < 5:
    raise ValueError("Too few clients for a meaningful client holdout.")
if df["is_declining_label"].nunique() != 2:
    raise ValueError("The target must contain both classes.")

# Split clients, not rows.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_CLIENT_FRACTION,
    random_state=RANDOM_STATE,
)
train_idx, test_idx = next(
    splitter.split(df, y=df["is_declining_label"], groups=df["client_id"])
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

assert set(train["client_id"]).isdisjoint(set(test["client_id"]))
assert train["is_declining_label"].nunique() == 2
assert test["is_declining_label"].nunique() == 2

print("Prepared rows:", len(df))
print("Training rows:", len(train))
print("Test rows:", len(test))
print("Training clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())
print("Client overlap:", len(set(train["client_id"]) & set(test["client_id"])))
print("Test proxy-label rate:", round(test["is_declining_label"].mean(), 4))

Prepared rows: 30000
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0
Test proxy-label rate: 0.511


## 3. Train + compare vs my baseline

I will reproduce the starter pipeline's transparent baseline score and compare it with both models on the same held-out clients. The baseline uses visibility, freshness, search position and content depth. The models use the permitted numeric and categorical features, excluding client and content IDs and fields used to construct the label.

The comparison table will report Precision@20, Precision@50 and the test-set proxy-label rate. No result will be claimed until the cells have actually run. If a model wins at one cutoff but loses at another, I will report both results rather than choosing only the favorable one.


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Match the starter pipeline's baseline score.
def percentile_rank(s):
    return pd.to_numeric(s, errors="coerce").fillna(0).rank(
        method="average", pct=True
    )

def normalize(s):
    s = pd.to_numeric(s, errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    ).fillna(0)
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(0.0, index=s.index)
    return (s - lo) / (hi - lo)

def baseline_score(frame):
    visibility = percentile_rank(np.log1p(frame["impressions_90d"]))
    freshness = percentile_rank(frame["days_since_last_update"])
    position = (
        (1 - normalize(frame["avg_position"].clip(lower=1, upper=50)))
        * visibility
        * (frame["avg_position"] > 0).astype(int)
    )
    depth = (
        (1 - percentile_rank(frame["word_count"])) * visibility
    )
    return (
        0.40 * visibility
        + 0.30 * freshness
        + 0.25 * position
        + 0.05 * depth
    ).clip(0, 1)

# Ensure required numeric fields exist and are numeric.
numeric_candidates = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "days_with_impressions",
    "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_candidates = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

# Use only columns available in the source.
numeric_features = [c for c in numeric_candidates if c in df.columns]
categorical_features = [c for c in categorical_candidates if c in df.columns]

# Do not use IDs, trend fields, labels, or product decisions as features.
forbidden = {
    "content_id", "client_id", "trend_direction", "trend_pct",
    "is_declining_label", "measurable_opportunity", "needs_ctr_fix",
    "quick_win", "action_type", "priority_score",
}
numeric_features = [c for c in numeric_features if c not in forbidden]
categorical_features = [c for c in categorical_features if c not in forbidden]

if not numeric_features and not categorical_features:
    raise ValueError("No eligible features found.")

X_train = train[numeric_features + categorical_features].copy()
X_test = test[numeric_features + categorical_features].copy()
y_train = train["is_declining_label"].astype(int)
y_test = test["is_declining_label"].astype(int)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

transformers = []
if numeric_features:
    transformers.append(("num", numeric_pipe, numeric_features))
if categorical_features:
    transformers.append(("cat", categorical_pipe, categorical_features))

preprocessor = ColumnTransformer(transformers=transformers)

models = {
    "Logistic Regression": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=RANDOM_STATE,
        )),
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=25,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
}

# Same test rows for baseline and both models.
baseline_test_score = baseline_score(test).to_numpy()

def precision_at_k(y, scores, k):
    order = np.argsort(-np.asarray(scores), kind="stable")[:min(k, len(y))]
    return float(np.asarray(y)[order].mean())

results = []
predictions = {}

base_row = {
    "method": "Rule baseline",
    "Precision@20": precision_at_k(y_test, baseline_test_score, 20),
    "Precision@50": precision_at_k(y_test, baseline_test_score, 50),
    "Test proxy-label rate": float(y_test.mean()),
}
results.append(base_row)
predictions["Rule baseline"] = baseline_test_score

for name, model in models.items():
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    predictions[name] = scores

    results.append({
        "method": name,
        "Precision@20": precision_at_k(y_test, scores, 20),
        "Precision@50": precision_at_k(y_test, scores, 50),
        "Test proxy-label rate": float(y_test.mean()),
    })

results_df = pd.DataFrame(results)
display(results_df.round(3))

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Random seed:", RANDOM_STATE)

,method,Precision@20,Precision@50,Test proxy-label rate
0,Rule baseline,0.35,0.32,0.511
1,Logistic Regression,0.75,0.74,0.511
2,Random Forest,0.65,0.58,0.511


Numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Categorical features: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']
Random seed: 42


## 4. Errors and interpretation

I will inspect examples where the model's ranking disagrees with the observed proxy label. High-ranked negative examples are false positives for decline-proxy prioritization; low-ranked positive examples are missed positives.

I will also inspect feature importance to understand which signals the Random Forest relies on. Importance describes how the model uses the data; it does not establish that a feature causes a page to decline.

The three example errors will be taken from the actual held-out test rows. Their interpretation will remain cautious because the label is a current-trend proxy, not a verified future outcome.


In [4]:
# Select the Random Forest for an interpretable first error review.
chosen_name = "Random Forest"
chosen_scores = predictions[chosen_name]

error_review = test[
    ["content_id", "client_id", "is_declining_label"]
].copy()
error_review["model_score"] = chosen_scores
error_review["rank"] = (
    error_review["model_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# False positives: ranked highly but proxy label is 0.
false_positives = error_review[
    error_review["is_declining_label"].eq(0)
].sort_values("rank").head(3).copy()
false_positives["error_type"] = "high-ranked, proxy-negative"

# False negatives: proxy label is 1 but ranked lower.
false_negatives = error_review[
    error_review["is_declining_label"].eq(1)
].sort_values("rank", ascending=False).head(3).copy()
false_negatives["error_type"] = "low-ranked, proxy-positive"

print("Potential false positives:")
display(false_positives)

print("Potential false negatives:")
display(false_negatives)

# Feature importance from the fitted Random Forest.
rf = models["Random Forest"]
feature_names = rf.named_steps["prep"].get_feature_names_out()
importance = rf.named_steps["model"].feature_importances_

importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importance,
    })
    .sort_values("importance", ascending=False)
)

print("Top 10 model features:")
display(importance_df.head(10).round(4))

# Save comparison and error review outputs.
from pathlib import Path

out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(out_dir / "ml08_model_comparison.csv", index=False)
error_review.to_csv(out_dir / "ml08_test_predictions.csv", index=False)
importance_df.to_csv(out_dir / "ml08_feature_importance.csv", index=False)

print("Saved comparison, test predictions, and feature importance.")

Potential false positives:


,content_id,client_id,is_declining_label,model_score,rank,error_type
22526,content_1d0963b56227,client_4e07408562,0,0.855565,2,"high-ranked, proxy-negative"
11061,content_0b47dae0c7f9,client_8527a891e2,0,0.854582,3,"high-ranked, proxy-negative"
22042,content_2ba626fea4d6,client_8527a891e2,0,0.851197,4,"high-ranked, proxy-negative"


Potential false negatives:


,content_id,client_id,is_declining_label,model_score,rank,error_type
1864,content_16f38acf0f26,client_e629fa6598,1,0.107599,6131,"low-ranked, proxy-positive"
27271,content_7bc32bc1df59,client_8527a891e2,1,0.141736,6102,"low-ranked, proxy-positive"
18423,content_77e2a54525b6,client_8527a891e2,1,0.168250,6089,"low-ranked, proxy-positive"


Top 10 model features:


,feature,importance
11,num__days_with_impressions,0.1440
5,num__impressions_90d,0.1078
16,num__avg_position,0.1029
13,num__content_age_days,0.0882
15,num__ctr,0.0325
56,cat__position_tier_top_3,0.0324
38,cat__age_tier_365+,0.0303
3,num__word_count,0.0302
18,num__scroll_rate,0.0296
4,num__char_count,0.0289


Saved comparison, test predictions, and feature importance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.